In [1]:
!python -V

Python 3.10.13


In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error

Q1 - Read the data for January. How many columns are there?

In [3]:
#In this assignment, we will use January data as train data and February data as validation data.
train_data = pd.read_parquet("./data/yellow_tripdata_2024-01.parquet")
val_data = pd.read_parquet("./data/yellow_tripdata_2024-02.parquet")

In [4]:
print(f"The number of columns of January data is {len(train_data.columns)}.")

The number of columns of January data is 19.


Q2 - What's the standard deviaton of the trips in January?

In [5]:
train_data["duration"] = ((train_data.tpep_dropoff_datetime - train_data.tpep_pickup_datetime).dt.total_seconds() / 60)

In [6]:
train_data_duration_mean = train_data.duration.sum() / len(train_data.duration)
train_data_duration_sum_of_squares = train_data.duration.apply(lambda x: (x - train_data_duration_mean)**2).sum()
train_data_duration_standard_deviation = (train_data_duration_sum_of_squares / (len(train_data.duration) - 1))**0.5
print(f"The standard deviation of the duration data is {train_data_duration_standard_deviation.round(2)}")

The standard deviation of the duration data is 34.85


Q3 - What fraction of the records left after you dropped the outliers?

In [7]:
num_rows_before_dropping_outliers = len(train_data.duration)
train_data = train_data[(train_data.duration <= 60) & (train_data.duration >= 1)]
print(f"The fraction of the records left after dropped outliers is {int((len(train_data) / num_rows_before_dropping_outliers)*100)}")

The fraction of the records left after dropped outliers is 97


Q4 - What's the dimensionality of this matrix (number of columns)?

In [8]:
train_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.0,19.800000
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,6.600000
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.0,17.916667
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.0,8.300000
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.0,6.100000


In [9]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

train_data[categorical] = train_data[categorical].astype(str)

train_dicts = train_data[categorical + numerical].to_dict(orient="records")

In [10]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer()
X_train = dv.fit_transform(train_dicts)

print(f"The dimensionality of this matrix (number of columns) is {X_train.shape[1]}")

The dimensionality of this matrix (number of columns) is 519


Q5 - What's the RMSE on train?

In [11]:
target = "duration"
y_train = train_data[target].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

rmse = root_mean_squared_error(y_pred, y_train)
print(f"The RMSE on train is {rmse}")

The RMSE on train is 7.952029670782532


Q5 - What's the RMSE on validation?

In [13]:
val_data["duration"] = ((val_data.tpep_dropoff_datetime - val_data.tpep_pickup_datetime).dt.total_seconds() / 60)
val_data = val_data[(val_data.duration <= 60) & (val_data.duration >= 1)]

In [14]:
val_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration
0,2,2024-02-01 00:04:45,2024-02-01 00:19:58,1.0,4.39,1.0,N,68,236,1,20.5,1.0,0.5,1.28,0.00,1.0,26.78,2.5,0.00,15.216667
1,2,2024-02-01 00:56:31,2024-02-01 01:10:53,1.0,7.71,1.0,N,48,243,1,31.0,1.0,0.5,9.00,0.00,1.0,45.00,2.5,0.00,14.366667
2,2,2024-02-01 00:07:50,2024-02-01 00:43:12,2.0,28.69,2.0,N,132,261,2,70.0,0.0,0.5,0.00,6.94,1.0,82.69,2.5,1.75,35.366667
3,1,2024-02-01 00:01:49,2024-02-01 00:10:47,1.0,1.10,1.0,N,161,163,1,9.3,3.5,0.5,2.85,0.00,1.0,17.15,2.5,0.00,8.966667
4,1,2024-02-01 00:37:35,2024-02-01 00:51:15,1.0,2.60,1.0,N,246,79,2,15.6,3.5,0.5,0.00,0.00,1.0,20.60,2.5,0.00,13.666667


In [15]:
val_data[categorical] = val_data[categorical].astype(str)
val_dicts = val_data[categorical + numerical].to_dict(orient="records")

: 

In [ ]:
X_val = dv.transform(val_dicts)
y_val = val_data[target].values

y_pred = lr.predict(X_val)

rmse = root_mean_squared_error(y_pred, y_val)
print(f"The RMSE on validation is {rmse}")